In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import h5py
import skimage as sk
%matplotlib inline

In [ ]:


channels = ['mCherry-Penta', 'GFP-Penta']
file_idx = 100

idx = 48

t_inds = list(range(40,80,1))
with h5py.File("/home/lag36/scratch/lag36/2022-08-19_MM-Ti5-lDE24/Growth_Division/kymograph/kymograph_" + str(file_idx) + ".hdf5", "r") as infile:  
    for ch in channels:
        img = infile[ch][idx][t_inds[0]]
        for t_ind in t_inds:
            img_ind = infile[ch][idx][t_ind]
            img = np.concatenate((img, img_ind), axis = 1)
        plt.imshow(img, cmap='gray')
        plt.show()
        
with h5py.File("/home/lag36/scratch/lag36/2022-08-19_MM-Ti5-lDE24/Growth_Division/fluorsegmentation/segmentation_" + str(file_idx) + ".hdf5", "r") as infile:
    seg_data = infile["data"][:]
    img = seg_data[idx][t_inds[0]]
    for t_ind in t_inds:
        img_ind = seg_data[idx][t_ind]
        img = np.concatenate((img, img_ind), axis = 1)
    plt.imshow(img)
    plt.show()

In [ ]:
nan_thr = 6 #3
percentile = 50
t_min = 0
t_max = 120


file_idx = 100

with h5py.File("/home/lag36/scratch/lag36/2022-08-19_MM-Ti5-lDE24/Growth_Division/kymograph/kymograph_" + str(file_idx) + ".hdf5", "r") as infile:
        gfp_data = infile["GFP-Penta"][:]
        
with h5py.File("/home/lag36/scratch/lag36/2022-08-19_MM-Ti5-lDE24/Growth_Division/fluorsegmentation/segmentation_" + str(file_idx) + ".hdf5", "r") as infile:
        seg_data = infile["data"][:]
    
idx = 1
t = 8

plt.imshow(seg_data[idx][t])
plt.show()
plt.imshow(gfp_data[idx][t])
plt.show()

rps = sk.measure.regionprops(seg_data[idx][t], gfp_data[idx][t])

degrees = [(rp.orientation/(2*np.pi))*360 for rp in rps]
centroids = [rp.centroid for rp in rps]

# Do it for the first one
i = 0
degree = degrees[0]
centroid = centroids[0]
masked_img = (seg_data[idx][t] == (i+1))*gfp_data[idx][t]
plt.imshow(masked_img)
plt.title('masked_img')
plt.show()

rotated_img = sk.transform.rotate(masked_img,-degree,center=centroid[::-1],preserve_range=True)
plt.imshow(rotated_img)
plt.title('Rotated img')
plt.show()
rotated_img[rotated_img==0.]=np.NaN

n_not_nan = np.sum(~np.isnan(rotated_img),axis=1)
plt.plot(n_not_nan, 'o')
plt.show()
nan_mask = n_not_nan>nan_thr
nan_masked_rotated_img = rotated_img[nan_mask]

percentile_trace = np.nanpercentile(nan_masked_rotated_img,percentile,axis=1)
sum_trace = np.nansum(nan_masked_rotated_img, axis = 1)
# percentile_traces.append(percentile_trace)
plt.plot(percentile_trace)
plt.title('median')
plt.show()
plt.plot(sum_trace)
plt.title('sum')

## Over time, for the mother cell only and a a fixed trench

In [ ]:
import dask.dataframe as dd
kymo_meta = dd.read_parquet('/home/lag36/scratch/lag36/2022-08-19_MM-Ti5-lDE24/Growth_Division/kymograph/metadata', engine='pyarrow', calculate_divisions=True)
sub_kymo_meta = kymo_meta[['File Index', 'File Trench Index', 'trenchid']].compute().set_index(['trenchid']).groupby('trenchid').apply(lambda x: x.iloc[0])

In [ ]:
nan_thr = 6 #3
percentile = 50
t_min = 0
t_max = 200


file_idx = 100
trench_indices = [48]

def make_fork_plot(nan_thr = 6, percentile = 50, t_min = 0, t_max = 200, trench_indices = [48], file_idx = 100):

    with h5py.File("/home/lag36/scratch/lag36/2022-08-19_MM-Ti5-lDE24/Growth_Division/kymograph/kymograph_" + str(file_idx) + ".hdf5", "r") as infile:
            gfp_data = infile["GFP-Penta"][:]

    with h5py.File("/home/lag36/scratch/lag36/2022-08-19_MM-Ti5-lDE24/Growth_Division/fluorsegmentation/segmentation_" + str(file_idx) + ".hdf5", "r") as infile:
            seg_data = infile["data"][:]

    percentile_traces = []
    # trench_indices = range(gfp_data.shape[0])

    for idx in trench_indices:
        for t in range(t_min,t_max):
        # plt.imshow(seg_data[idx][t])
        # plt.show()
        # plt.imshow(gfp_data[idx][t])
        # plt.show()

            rps = sk.measure.regionprops(seg_data[idx][t], gfp_data[idx][t])

            degrees = [(rp.orientation/(2*np.pi))*360 for rp in rps]
            centroids = [rp.centroid for rp in rps]

            # Do it for the first one
            # i = 0
            for i,degree in enumerate(degrees):
                degree = degrees[0]
                centroid = centroids[0]
                masked_img = (seg_data[idx][t] == (i+1))*gfp_data[idx][t]
                # plt.imshow(masked_img)
                # plt.title('masked_img')
                # plt.show()

                rotated_img = sk.transform.rotate(masked_img,-degree,center=centroid[::-1],preserve_range=True)
                # plt.imshow(rotated_img)
                # plt.title('Rotated img')
                # plt.show()
                rotated_img[rotated_img==0.]=np.NaN

                n_not_nan = np.sum(~np.isnan(rotated_img),axis=1)
                # plt.plot(n_not_nan, 'o')
                # plt.show()
                nan_mask = n_not_nan>nan_thr
                nan_masked_rotated_img = rotated_img[nan_mask]

                percentile_trace = np.nanpercentile(nan_masked_rotated_img,percentile,axis=1)
                sum_trace = np.nansum(nan_masked_rotated_img, axis = 1)
                # percentile_traces.append(percentile_trace)
                # plt.plot(percentile_trace)
                # plt.title('median')
                # plt.show()
                # plt.plot(sum_trace)
                # plt.title('sum')
                percentile_traces.append(percentile_trace)

    percentile_traces = np.array(percentile_traces)
    trace_lens = np.array([len(trace) for trace in percentile_traces])
    trace_bins = np.sort(np.unique(trace_lens))
    med_traces = []
    for trace_len in trace_bins:
        trace_group = np.stack(percentile_traces[trace_lens==trace_len])
        med_trace = np.nanmedian(trace_group,axis=0)
        med_traces.append(med_trace)
    max_trace_len = np.max(trace_bins)
    trace_dif = max_trace_len-trace_bins
    padded_traces = np.array([np.pad(trace,((trace_dif[i]//2)+trace_dif[i]%2,trace_dif[i]//2),constant_values=np.NaN) for i,trace in enumerate(med_traces)])

    plt.imshow(padded_traces[:],vmin=300,vmax=500)

In [ ]:
make_fork_plot(nan_thr = 6, percentile = 50, t_min = 0, t_max = 200, trench_indices = [48], file_idx = 100)

In [ ]:
nan_thr = 3
percentile = 99
t_min = 0
t_max = 60

percentile_traces = []

file_indices = np.sort([int(item.split("_")[1].split(".")[0]) for item in os.listdir("/home/de64/scratch/de64/sync_folder/2022-01-23_DE511_test/kymograph/") if 'kymograph' in item])

for file_idx in file_indices:
    if file_idx not in remaining_file_idx_lookup.keys():
        continue
    trench_indices = remaining_file_idx_lookup[file_idx]
    with h5py.File("/home/de64/scratch/de64/sync_folder/2022-02-11_DE524_rne/kymograph/kymograph_" + str(file_idx) + ".hdf5", "r") as infile:
        yfp_data = infile["YFP"][:]
    with h5py.File("/home/de64/scratch/de64/sync_folder/2022-02-11_DE524_rne/fluorsegmentation/segmentation_" + str(file_idx) + ".hdf5", "r") as infile:
        seg_data = infile["data"][:]

    for idx in trench_indices:
        
        for t in range(t_min,t_max):
            yfp_data[idx,t] = sk.filters.gaussian(yfp_data[idx,t],sigma=2.,preserve_range=True)
            
            rps = sk.measure.regionprops(seg_data[idx][t],yfp_data[idx][t])
            degrees = [(rp.orientation/(2*np.pi))*360 for rp in rps]
            centroids = [rp.centroid for rp in rps]

            for i,degree in enumerate(degrees):
                centroid = centroids[i]
                masked_img = (seg_data[idx][t]==(i+1))*yfp_data[idx][t]
                rotated_img = sk.transform.rotate(masked_img,degree,center=centroid[::-1],preserve_range=True)
                rotated_img[rotated_img==0.]=np.NaN

                n_not_nan = np.sum(~np.isnan(rotated_img),axis=1)
                nan_mask = n_not_nan>nan_thr
                nan_masked_rotated_img = rotated_img[nan_mask]

                percentile_trace = np.nanpercentile(nan_masked_rotated_img,percentile,axis=1)
                percentile_traces.append(percentile_trace)
percentile_traces = np.array(percentile_traces)
trace_lens = np.array([len(trace) for trace in percentile_traces])
trace_bins = np.sort(np.unique(trace_lens))
med_traces = []
for trace_len in trace_bins:
    trace_group = np.stack(percentile_traces[trace_lens==trace_len])
    med_trace = np.nanmedian(trace_group,axis=0)
    med_traces.append(med_trace)
max_trace_len = np.max(trace_bins)
trace_dif = max_trace_len-trace_bins
padded_traces = np.array([np.pad(trace,((trace_dif[i]//2)+trace_dif[i]%2,trace_dif[i]//2),constant_values=np.NaN) for i,trace in enumerate(med_traces)])

plt.imshow(padded_traces[:80],vmin=1350,vmax=1800)